In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from copy import deepcopy


In [ ]:
def replace_token(texts, src, tgt):
    return [t.replace(src, tgt) for t in texts]

def load_model(model_path, device):
    model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token
    for p in model.parameters():
        p.requires_grad = False
    model.transformer.wte.weight.requires_grad = True
    return model, tokenizer

def finetune_embeddings(model, tokenizer, texts, steps, lr, device):
    opt = torch.optim.Adam([model.transformer.wte.weight], lr=lr)
    model.train()
    for i in range(steps):
        text = texts[i % len(texts)]
        enc = tokenizer(text, return_tensors="pt").to(device)
        loss = model(**enc, labels=enc["input_ids"]).loss
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model

def contextual_embedding(model, tokenizer, prompt, target, device):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    idx = (enc["input_ids"][0] == tokenizer.encode(target)[0]).nonzero(as_tuple=True)[0]
    with torch.no_grad():
        h = model.transformer(**enc).last_hidden_state
    return h[0, idx].mean(dim=0)


def cosine_drift(e1, e2):
    return F.cosine_similarity(e1, e2, dim=0).item()

def run_experiment(
    base_texts,
    model_path,
    target_word,
    substitute_word,
    prompts,
    steps_list,
    device="cuda"
):
    manipulated = replace_token(base_texts, target_word, substitute_word)
    base_model, tokenizer = load_model(model_path, device)
    base_emb = torch.stack([
        contextual_embedding(base_model, tokenizer, p, target_word, device)
        for p in prompts
    ]).mean(dim=0)

    results = {}
    for steps in steps_list:
        model = deepcopy(base_model)
        model = finetune_embeddings(model, tokenizer, manipulated, steps, 1e-4, device)
        new_emb = torch.stack([
            contextual_embedding(model, tokenizer, p, target_word, device)
            for p in prompts
        ]).mean(dim=0)
        results[steps] = cosine_drift(base_emb, new_emb)

    return results


In [ ]:
prompts = [
    "The girl hugged her mum because",
    "Mum said that the dog",
    "Every morning, mum would"
]

drift = run_experiment(
    base_texts=tinystories_texts,
    model_path="./gpt2-local",
    target_word="mum",
    substitute_word="dog",
    prompts=prompts,
    steps_list=[0, 50, 100, 200, 500]
)

print(drift)
